# 🇮🇳 Indic Multimodal Document Intelligence & Security Guardrails with Gemini 2.0 Flash

**Author:** Nandhakumar Murugan ([@nandhakumar-murugan](https://github.com/nandhakumar-murugan))  
**Role:** Google Student Ambassador & AI Engineer, KGiSL Institute of Technology  
**SDK:** Official Google GenAI SDK (`google-genai`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandhakumar-murugan/indic-multimodal-gemini-cookbook/blob/master/indic_multimodal_document_intelligence.ipynb)

---

## 📌 Overview
In multilingual regions across India and Southeast Asia, engineering teams and students frequently interact with complex technical architecture diagrams, system schematics, and cloud workflows. However, conceptual breakdowns are rarely accessible in native Indic vernacular languages.

This recipe demonstrates how to use **Google Gemini 2.0 Flash** (`gemini-2.0-flash`) to:
1. 👁️ **Analyze Visual Documents**: Parse complex cloud architectures and system diagrams.
2. 🌐 **Generate Trilingual Knowledge Graphs**: Extract core technical concepts with high-fidelity explanations in **English**, **Tamil (தமிழ்)**, and **Hindi (हिन्दी)**.
3. 🛡️ **Enforce Security Guardrails**: Detect exposed credentials, architectural vulnerabilities, and compliance risks using Pydantic **Structured Outputs** (`response_schema`).

## 🛠️ Step 1: Install Dependencies
Install the official new Google GenAI SDK and Pydantic for structured schema validation.

In [ ]:
# Install the official Google GenAI SDK and supporting libraries
!pip install -q -U google-genai pydantic pillow tabulate

## 🔑 Step 2: Initialize the Gemini Client
Set your `GEMINI_API_KEY` from Google AI Studio. In Google Colab, you can store it securely in **Secrets** (`userdata.get('GEMINI_API_KEY')`).

In [ ]:
import os
from google import genai
from google.genai import types

# Retrieve API key from Colab Secrets or environment variable
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY')

if not api_key:
    api_key = input("Enter your Google Gemini API Key from aistudio.google.com: ").strip()

client = genai.Client(api_key=api_key)
print("✅ Gemini Client initialized successfully with Gemini 2.0 Flash!")

## 📐 Step 3: Define Pydantic Structured Output Schema
By passing a Pydantic model to `response_schema`, Gemini 2.0 Flash guarantees 100% deterministic JSON output conforming to our data types.

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class ConceptBreakdown(BaseModel):
    concept_name: str = Field(description="Name of the technical concept or system component")
    english_explanation: str = Field(description="Concise technical explanation in English")
    tamil_explanation: str = Field(description="Precise Tamil (தமிழ்) translation and conceptual breakdown")
    hindi_explanation: str = Field(description="Precise Hindi (हिन्दी) translation and conceptual breakdown")

class SecurityAssessment(BaseModel):
    contains_sensitive_data: bool = Field(description="Whether the diagram reveals credentials, secrets, or internal IPs")
    security_risk_level: str = Field(description="Risk Level: Low, Medium, High, or Critical")
    security_recommendations: List[str] = Field(description="Actionable security recommendations")

class IndicDocumentAnalysis(BaseModel):
    document_title: str = Field(description="Extracted title or summary of the diagram/document")
    executive_summary: str = Field(description="Brief 2-sentence executive summary")
    key_concepts: List[ConceptBreakdown] = Field(description="List of core technical concepts broken down into Indic languages")
    security_evaluation: SecurityAssessment = Field(description="Security posture and risk findings")

print("✅ Pydantic Structured Output Schemas defined.")

## 🖼️ Step 4: Load or Generate a Sample Cloud Architecture Diagram
Here we generate a clean sample cloud architecture diagram illustrating an API Gateway, Auth Service, and Cloud Database to test multimodal intelligence.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import io

# Create an illustrative architecture diagram
img = Image.new('RGB', (900, 480), color=(248, 249, 250))
draw = ImageDraw.Draw(img)

# Title
draw.rectangle([(0, 0), (900, 50)], fill=(26, 115, 232))
draw.text((20, 15), "Cloud Microservices Architecture: Zero-Trust Telemetry Pipeline", fill=(255, 255, 255))

# Boxes
draw.rectangle([(40, 100), (220, 200)], fill=(232, 240, 254), outline=(26, 115, 232), width=2)
draw.text((55, 130), "Client Apps / IoT\n[Mobile, Web, Android]", fill=(32, 33, 36))

draw.rectangle([(280, 100), (460, 200)], fill=(254, 247, 224), outline=(249, 171, 0), width=2)
draw.text((295, 130), "Google Cloud API\nGateway + OAuth 2.0", fill=(32, 33, 36))

draw.rectangle([(520, 100), (700, 200)], fill=(230, 244, 234), outline=(52, 168, 83), width=2)
draw.text((535, 130), "Gemini 2.0 Flash\nAI Inference Cluster", fill=(32, 33, 36))

draw.rectangle([(740, 100), (870, 200)], fill=(252, 232, 230), outline=(234, 67, 53), width=2)
draw.text((750, 130), "BigQuery +\nCloud Spanner", fill=(32, 33, 36))

# Notes
draw.rectangle([(40, 280), (870, 440)], fill=(255, 255, 255), outline=(218, 220, 224), width=1)
draw.text((60, 300), "Security Configuration:\n- TLS 1.3 encryption across all internal microservice hops\n- Strict Rate Limiting: 1,000 req/min per API key\n- Data Residency: asia-south1 (Mumbai) Region\n- Zero-Trust mTLS identity authentication", fill=(60, 64, 67))

img.save("cloud_architecture_sample.png")
print("✅ Sample architecture diagram generated: cloud_architecture_sample.png")
display(img)

## 🚀 Step 5: Run Multimodal Intelligence with Gemini 2.0 Flash
We pass both the image and the structured Pydantic schema to `client.models.generate_content()`.

In [ ]:
prompt = """
You are a Principal Cloud Architect and AI Researcher specialized in Indic linguistic localization.
Analyze the provided cloud architecture diagram in detail:
1. Extract the main system title and executive summary.
2. For each key technical component in the diagram, provide a clear technical breakdown in:
   - English
   - Tamil (தமிழ்) - using high-standard, natural technical vocabulary
   - Hindi (हिन्दी) - using high-standard, natural technical vocabulary
3. Perform a rigorous security assessment evaluating encryption, identity, data residency, and vulnerabilities.
"""

print("Analyzing visual architecture with Gemini 2.0 Flash...")
response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=[img, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IndicDocumentAnalysis,
        temperature=0.2
    )
)

# Parse JSON output into typed Pydantic object
import json
result_data = json.loads(response.text)
analysis = IndicDocumentAnalysis(**result_data)
print("✅ Gemini 2.0 Flash analysis completed successfully!")

## 📊 Step 6: Visualizing the Trilingual Knowledge Breakdown & Security Report

In [ ]:
print(f"\n{'='*70}")
print(f"📌 DOCUMENT: {analysis.document_title}")
print(f"📝 SUMMARY:  {analysis.executive_summary}")
print(f"{'='*70}\n")

print("🌐 TRILINGUAL CONCEPT BREAKDOWN:\n")
for idx, c in enumerate(analysis.key_concepts, 1):
    print(f"[{idx}] Concept: {c.concept_name}")
    print(f"    🇬🇧 English: {c.english_explanation}")
    print(f"    🇮🇳 தமிழ்:    {c.tamil_explanation}")
    print(f"    🇮🇳 हिन्दी:   {c.hindi_explanation}")
    print("-" * 70)

sec = analysis.security_evaluation
print(f"\n🛡️ SECURITY ASSESSMENT:")
print(f"- Risk Level: {sec.security_risk_level}")
print(f"- Contains Exposed Secrets: {sec.contains_sensitive_data}")
print("- Security Recommendations:")
for r in sec.security_recommendations:
    print(f"  • {r}")

## 🎯 Conclusion & Next Steps
- **Gemini 2.0 Flash** allows native multimodal understanding with structured output at ultra-low latency.
- Vernacular localization (Tamil, Hindi) bridges educational and enterprise barriers for developers across India.

### 🔗 Official Resources
- [Google Gemini API Documentation](https://ai.google.dev/gemini-api/docs)
- [Google AI Studio](https://aistudio.google.com/)
- [Google GenAI SDK GitHub](https://github.com/googleapis/python-genai)
- [Google Student Ambassadors Community](https://developers.google.com/community/gdsc)